# Day 4 — 모델 양자화 (INT8 / INT4)

Day 1에서는 `Qwen/Qwen2.5-0.5B-Instruct`를 fp16/bf16으로 그대로 로드해서 사용했습니다. 오늘은 **양자화(quantization)** — 가중치를 더 적은 비트로 표현해서 메모리와 속도를 개선하는 기법 — 를 다룹니다.

핵심 질문:
1. fp16 → int8 → int4로 갈수록 모델 크기와 메모리 사용량이 어떻게 줄어드는가?
2. Jetson Orin Nano(sm_87, aarch64, 8GB 통합메모리)에서 **실제로 동작하는** 양자화 방법은 무엇인가?
3. 속도/메모리를 아끼는 대신 무엇을 잃는가 (출력 품질)?

> **먼저 냉정하게 짚고 갈 것**: LLM 양자화 생태계(bitsandbytes, AutoGPTQ, AutoAWQ 등)의 사전 컴파일 바이너리는 거의 전부 x86_64 + 서버 GPU(sm_75/80/86/90) 기준으로 배포됩니다. Jetson의 sm_87 aarch64 조합은 이 매트릭스에서 종종 빠져 있습니다. 아래에서 실제로 무엇이 되고 안 되는지 하나씩 확인합니다.

## 1. bitsandbytes를 pip으로 그냥 설치하면 어떻게 되는가 (기대와 현실)

`transformers`의 `load_in_8bit=True` / `load_in_4bit=True` 옵션은 내부적으로 **bitsandbytes**를 사용합니다. 이론상 가장 쉬운 방법이지만, Jetson에서는 함정이 있습니다.

GitHub 이슈([bitsandbytes-foundation/bitsandbytes #1930](https://github.com/bitsandbytes-foundation/bitsandbytes/issues/1930))에 정확히 이 케이스가 보고되어 있습니다: bitsandbytes가 PyPI에 배포하는 **aarch64 휠은 sm_75/sm_80/sm_90용 CUDA 커널만 사전 컴파일**되어 있고 **sm_87(Jetson Orin)은 빠져 있습니다**. 즉:

- `pip install bitsandbytes` 자체는 aarch64에서도 **설치는 됩니다** (에러 없이 끝남).
- 하지만 실제로 8bit/4bit 레이어를 GPU에서 실행하려는 순간, sm_87용 커널이 없어서 **첫 CUDA 커널 호출 시 런타임 에러**가 납니다.
- 공식 우회법은 (a) 소스에서 `TORCH_CUDA_ARCH_LIST=8.7`로 직접 빌드하거나, (b) Jetson AI Lab이 제공하는 커뮤니티 휠 인덱스(`pypi.jetson-ai-lab.io`)를 쓰는 것인데, 둘 다 이 7일 스프린트의 범위를 벗어나는 빌드 환경 설정이 필요합니다.

아래 셀에서 실제로 무슨 일이 일어나는지 직접 확인해봅니다 (에러가 나는 것 자체가 학습 포인트입니다).

In [ ]:
import subprocess, sys

# bitsandbytes 설치 시도 (실패해도 무해함 — 학습 목적)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"],
    capture_output=True, text=True
)
print("설치 종료 코드:", result.returncode)
print(result.stdout[-1000:])
print(result.stderr[-1000:])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    import bitsandbytes as bnb
    print("bitsandbytes import 성공, 버전:", bnb.__version__)

    # 함정: 작은 랜덤 텐서로 Linear8bitLt를 테스트하면 '성공'처럼 보일 수 있습니다.
    x = torch.randn(64, 64, device="cuda", dtype=torch.float16)
    layer = bnb.nn.Linear8bitLt(64, 64, has_fp16_weights=False).to("cuda")
    out = layer(x)
    print("(참고) 64x64 토이 텐서 8bit Linear는 에러 없이 실행됨 — 하지만 이것만으로는 안심할 수 없습니다.")

    # 진짜 검증: 실제 모델을 8bit로 로드해서 generate()까지 돌려봅니다.
    MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model_8bit = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="cuda")
    print("8bit 모델 로드 성공, GPU 메모리:", torch.cuda.memory_allocated() / 1024**3, "GB")

    chat_str = tokenizer.apply_chat_template(
        [{"role": "user", "content": "안녕"}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat_str, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model_8bit.generate(**inputs, max_new_tokens=10)
    print("실제 모델 8bit generate() 성공! 이 환경에서는 bitsandbytes가 완전히 동작합니다.")
except ImportError as e:
    print("bitsandbytes import 자체가 실패:", e)
except Exception as e:
    print("실제 모델 generate()에서 실패:", type(e).__name__, str(e)[:300])

**실측 결과**: 위 셀을 실제로 돌려보면 흥미로운 패턴이 나옵니다 — **64x64 토이 텐서로 만든 `Linear8bitLt`는 에러 없이 통과**하지만, **실제 모델을 `BitsAndBytesConfig(load_in_8bit=True)`로 로드해서 `generate()`를 호출하면 `cuBLAS API failed with status 15`로 실패**합니다.

이게 이 섹션의 진짜 교훈입니다: **작은 토이 예제로 "라이브러리가 동작하는지" 확인하는 것은 신뢰할 수 없습니다.** 실제 모델의 실제 연산 크기/shape에서만 드러나는 실패가 있기 때문입니다. sm_87용 사전 컴파일 커널이 없다는 근본 원인([issue #1930](https://github.com/bitsandbytes-foundation/bitsandbytes/issues/1930))은 동일하지만, 실패가 나타나는 지점은 예상보다 더 뒤쪽(실제 forward pass 도중)이었습니다.

**결론: bitsandbytes 기반 `load_in_8bit`/`load_in_4bit`는 이 하드웨어에서 기본 pip 설치만으로는 신뢰할 수 없습니다.** 아래에서 세 가지 실전 대안을 각각 시도합니다.

1. **`torch.quantization`의 dynamic quantization** — PyTorch 코어 내장, 별도 패키지 불필요. 단, **CPU 전용**입니다 (Jetson GPU는 지원 안 함). 장점: 100% 확실히 동작.
2. **`optimum-quanto`** — 순수 PyTorch 텐서 서브클래스 기반 양자화 라이브러리. int8은 표준 torch 연산으로 폴백해 안정적으로 동작하지만, int4는 뒤에서 보듯 별도 위험이 있습니다.
3. **사전 양자화된 GGUF 체크포인트 + `llama-cpp-python`** — 양자화를 직접 수행하는 게 아니라, 이미 양자화된 모델을 그대로 가져와 CUDA(sm_87) 백엔드로 빌드된 llama.cpp로 로드하는 방식. 실무에서는 오히려 이 경로가 Jetson급 엣지 디바이스의 표준 배포 방식입니다.

## 2. 베이스라인: fp16 모델 크기와 메모리 사용량

먼저 비교 기준이 될 fp16 모델의 디스크 크기와 GPU 메모리 사용량을 측정합니다.

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

def gpu_mem_gb():
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated() / 1024**3

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda")

n_params = sum(p.numel() for p in model_fp16.parameters())
fp16_mem = gpu_mem_gb()
print(f"파라미터 수: {n_params/1e6:.1f}M")
print(f"이론적 fp16 크기: {n_params*2/1024**3:.3f} GB (파라미터당 2바이트)")
print(f"실제 GPU 메모리 사용량(fp16 로드 직후): {fp16_mem:.3f} GB")

# 아래 셀들에서 재사용할 입력 준비 (apply_chat_template은 tokenize=False로 문자열만 받고,
# 실제 텐서 변환은 tokenizer()에게 맡기는 게 이 환경에서 가장 안전합니다 —
# apply_chat_template(..., return_tensors="pt")를 바로 쓰면 버전에 따라
# 기대와 다른 객체가 반환되어 .shape 접근이 실패할 수 있습니다.)
prompt = "우주 탐사가 인류에게 중요한 이유를 두 문장으로 설명해줘."
messages = [{"role": "user", "content": prompt}]
chat_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_str, return_tensors="pt").to("cuda")

In [ ]:
torch.cuda.synchronize(); t0 = time.time()
with torch.no_grad():
    out_fp16 = model_fp16.generate(**inputs, max_new_tokens=80, do_sample=False)
torch.cuda.synchronize(); fp16_time = time.time() - t0

text_fp16 = tokenizer.decode(out_fp16[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"fp16 생성 시간: {fp16_time:.2f}초")
print("출력:", text_fp16)

## 3. 방법 A — `torch.quantization` dynamic quantization (CPU 전용)

PyTorch 코어에 내장된 가장 확실한 방법입니다. `nn.Linear` 레이어의 가중치를 int8로 변환하되, **연산 자체는 CPU에서** 이루어집니다 (Jetson GPU의 CUDA 텐서 코어를 int8 dynamic quant용으로 지원하지 않기 때문). 따라서 이건 "GPU를 더 빠르게"가 아니라 "메모리에 안 올라가는 상황에서 CPU로라도 돌리기" 또는 "양자화 개념 자체를 안전하게 실습하기" 용도로 이해하는 게 정확합니다.

In [ ]:
import copy

# CPU로 복사 후 dynamic quantization 적용 (GPU 텐서에는 적용 불가)
model_cpu_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)

model_int8_dynamic = torch.quantization.quantize_dynamic(
    model_cpu_fp32, {torch.nn.Linear}, dtype=torch.qint8
)

def state_dict_size_gb(m):
    total_bytes = 0
    for p in m.state_dict().values():
        if hasattr(p, "nelement"):
            total_bytes += p.nelement() * p.element_size()
    return total_bytes / 1024**3

print(f"fp32 원본 state_dict 크기: {state_dict_size_gb(model_cpu_fp32):.3f} GB")
print(f"dynamic int8 quant 후 state_dict 크기: {state_dict_size_gb(model_int8_dynamic):.3f} GB")

In [ ]:
inputs_cpu = {k: v.to("cpu") for k, v in inputs.items()}

t0 = time.time()
with torch.no_grad():
    out_int8 = model_int8_dynamic.generate(**inputs_cpu, max_new_tokens=80, do_sample=False)
int8_dynamic_time = time.time() - t0

text_int8 = tokenizer.decode(out_int8[0][inputs_cpu["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"CPU int8 dynamic quant 생성 시간: {int8_dynamic_time:.2f}초 (CPU 실행이라 GPU fp16보다 느릴 수 있음)")
print("출력:", text_int8)

# 메모리 확보를 위해 정리
del model_cpu_fp32, model_int8_dynamic
import gc; gc.collect()

**관찰**: dynamic quantization은 Linear 레이어 가중치 크기를 약 4배(fp32→int8) 줄이지만, 실행이 CPU에서 일어나므로 Jetson의 GPU 가속(sm_87 Ampere 코어)을 전혀 활용하지 못합니다. Orin Nano의 8코어 Cortex-A78AE CPU는 GPU 대비 LLM 추론이 훨씬 느리기 때문에, 이 방법은 **"메모리가 부족해서 GPU에 못 올릴 때의 최후 수단"** 정도로 이해하면 됩니다.

## 4. 방법 B — `optimum-quanto`로 GPU에서 int8/int4 양자화

`optimum-quanto`는 순수 PyTorch 텐서 서브클래스(`QTensor`)로 양자화를 구현하기 때문에, bitsandbytes처럼 아키텍처별 사전 컴파일 CUDA 커널에 의존하지 않습니다. sm_87에서 최적화된 커스텀 커널이 없더라도 표준 torch 연산으로 폴백해서 **일단 동작은 하는 것**을 최우선 목표로 설계되어 있습니다. GPU 메모리 절감 효과를 그대로 누리면서 CUDA에서 실행할 수 있는 현실적인 선택지입니다.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optimum-quanto"],
                        capture_output=True, text=True)
print(result.returncode)
print(result.stdout[-500:], result.stderr[-500:])

In [ ]:
from optimum.quanto import qint8, qint4, quantize, freeze

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# fp16 모델을 다시 로드한 뒤 quanto로 int8 양자화 적용
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda")
quantize(base_model, weights=qint8)
freeze(base_model)

int8_quanto_mem = gpu_mem_gb()
print(f"optimum-quanto int8 적용 후 GPU 메모리: {int8_quanto_mem:.3f} GB (fp16 대비 {fp16_mem:.3f} GB)")

In [ ]:
torch.cuda.synchronize(); t0 = time.time()
with torch.no_grad():
    out_quanto8 = base_model.generate(**inputs, max_new_tokens=80, do_sample=False)
torch.cuda.synchronize(); quanto8_time = time.time() - t0

text_quanto8 = tokenizer.decode(out_quanto8[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"quanto int8 생성 시간: {quanto8_time:.2f}초 (참고: fp16은 {fp16_time:.2f}초)")
print("출력:", text_quanto8)

del base_model
torch.cuda.empty_cache(); gc.collect()

# 실측: GPU 메모리 약 0.72GB (fp16의 ~1GB 대비 소폭 감소), 생성 60토큰 기준 약 10.6초
# (fp16보다 오히려 느립니다 — sm_87 전용 최적화 커널이 없어 양자화 텐서 연산이
#  일반 torch 폴백 경로로 도는 오버헤드 때문입니다. 메모리 절감이 목적일 때만 의미 있는 트레이드오프입니다.)

### ⚠️ int4는 여기서 실행하지 않습니다 — 실제로 기기가 멈췄습니다

처음에는 int8과 똑같은 방식으로 `quantize(model, weights=qint4)`를 시도했습니다. 그 과정에서 실제로 겪은 일:

1. **1차 시도**: `RuntimeError: Ninja is required to load C++ extensions`. `optimum-quanto`의 int4 경로는 AWQ 스타일 CUDA 커널을 **최초 실행 시점에 즉석으로(JIT) 컴파일**하는데, 이 컴파일에 `ninja` 빌드 도구가 필요했습니다.
2. **`pip install ninja` 후 재시도**: 이번엔 컴파일이 시작됐지만, `optimum-quanto`가 필요한 커널 하나만 컴파일하는 게 아니라 **AWQ/Marlin/GPTQ 등 CUDA 확장 라이브러리 전체를 한꺼번에 병렬로(cudafe++/cc1plus 프로세스 여러 개) 컴파일**했습니다. 각 컴파일 프로세스가 1GB 이상의 메모리를 먹으면서 **8GB 통합 메모리 + 2GB 스왑을 전부 소진**시켰고, load average가 20 가까이 치솟으며 **SSH 접속조차 몇 분간 안 될 정도로 기기 전체가 멈췄습니다.**

**교훈**: 8GB급 통합 메모리 엣지 기기에서 "필요할 때 CUDA 커널을 즉석 컴파일"하는 라이브러리는 컴파일 자체가 별도의 무거운 리소스 소비 작업이라는 걸 잊기 쉽습니다. 데스크탑/서버 GPU 환경(RAM 32GB+)에서는 무해한 일이 엣지 기기에서는 시스템 전체를 멈추는 사고로 이어질 수 있습니다.

**이 노트북에서는 int4 quanto를 실행하지 않습니다.** 정말 시도하고 싶다면 최소한 아래 안전장치를 먼저 적용하세요 (이 노트북 범위 밖):
- `MAX_JOBS=1` 환경변수로 병렬 컴파일 job 수를 1로 제한
- `tegrastats`로 메모리를 모니터링하며 실행
- 가능하면 스왑을 늘리거나, 컴파일이 끝난 사전 빌드 wheel을 다른 기기에서 만들어 옮겨오기

대신 int4가 정말 필요한 상황(7B급 이상 모델)에는 다음 섹션의 **사전 양자화된 GGUF + llama.cpp** 경로를 쓰는 게 실용적입니다 — 컴파일 없이 이미 만들어진 양자화 가중치 파일을 그대로 다운로드해서 쓰는 방식이기 때문입니다.

In [ ]:
# 실행하지 않는 참고 코드입니다 (위 markdown 참고 — 이 하드웨어에서 컴파일이 시스템을 멈출 위험이 있어
# 이 노트북에서는 검증하지 않았습니다). 안전장치를 갖춘 뒤 직접 시도해보고 싶다면 아래를 참고하세요.
#
# import os
# os.environ["MAX_JOBS"] = "1"  # 컴파일 병렬도 제한 (필수 — 안 하면 메모리 폭발 위험)
# from optimum.quanto import qint4, quantize, freeze
#
# base_model_4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda")
# quantize(base_model_4, weights=qint4)  # 최초 실행 시 CUDA 커널 JIT 컴파일 발생 (수 분 소요 가능)
# freeze(base_model_4)
# ...

print("이 셀은 실행하지 않습니다 - 위 markdown에서 이유를 설명합니다.")

## 5. 방법 C — 사전 양자화된 GGUF 체크포인트 (llama.cpp 경로)

실무에서 Jetson 같은 엣지 디바이스에 LLM을 배포할 때 가장 널리 쓰이는 방식은 사실 **양자화를 직접 수행하는 것이 아니라, 커뮤니티가 이미 GGUF 포맷(Q4_K_M, Q8_0 등)으로 양자화해 Hugging Face Hub에 올려둔 체크포인트를 그대로 다운로드해서 llama.cpp(sm_87 CUDA 빌드)로 로드**하는 것입니다.

이 경로는 `transformers`가 아니라 별도 `llama-cpp-python` 패키지(내부적으로 llama.cpp를 CUDA 지원으로 컴파일)를 사용합니다. 이 노트북 환경에는 사전 설치되어 있지 않으므로, **실제로 CUDA 가속까지 쓰려면 소스에서 `-DGGML_CUDA=ON`으로 빌드해야 하는 별도 설치 단계**가 필요합니다 (pip 바이너리 휠은 보통 CPU 전용으로 빌드되어 있음). 여기서는 설치/빌드 자체는 별도 셸 작업으로 남겨두고, 워크플로우와 기대 효과만 코드로 스케치합니다.

In [ ]:
# 참고용 스니펫 (실제 실행에는 CUDA로 빌드된 llama-cpp-python 필요):
#
# CMAKE_ARGS="-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=87" pip install llama-cpp-python --no-cache-dir
#
# from huggingface_hub import hf_hub_download
# from llama_cpp import Llama
#
# gguf_path = hf_hub_download(
#     repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
#     filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
# )
# llm = Llama(model_path=gguf_path, n_gpu_layers=-1, n_ctx=2048)  # n_gpu_layers=-1: 전 레이어 GPU 오프로드
# result = llm.create_chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=80)
# print(result["choices"][0]["message"]["content"])

print("이 셀은 실행하지 않는 설계도입니다 — 아래 markdown에서 왜 그런지, 그리고 언제 이 경로를 선택해야 하는지 설명합니다.")

**언제 GGUF+llama.cpp 경로를 쓰는가**: 0.5B~3B급의 작은 모델을 이번 노트북처럼 실험/학습 목적으로 다룰 때는 `transformers` + `optimum-quanto` 조합으로 충분합니다. 하지만 실제 배포에서 7B급 이상 모델을 Jetson Orin Nano의 8GB에 욱여넣어야 하는 상황이라면, Q4_K_M GGUF + llama.cpp(sm_87 CUDA 빌드)가 사실상 표준 경로입니다 (예: Llama 3.1 8B Q4_K_M ≈ 4.7GB 파일 + KV 캐시 포함 약 6GB). 이 스프린트에서는 시간 관계상 빌드 단계는 생략하지만, 다음 단계로 반드시 시도해볼 만한 실전 경로임을 기억해 두세요.

## 6. 종합 비교

지금까지 측정한 값들을 하나의 표로 정리합니다. (숫자는 실행 환경에 따라 달라지므로, 아래 셀에서 실제 측정값을 다시 채워 표를 만듭니다.)

In [ ]:
import pandas as pd

rows = [
    {"방법": "fp16 (베이스라인)", "실행 위치": "GPU", "GPU 메모리(GB)": round(fp16_mem, 3), "생성 시간(초)": round(fp16_time, 2), "출력": text_fp16[:40]},
    {"방법": "torch dynamic int8 (CPU)", "실행 위치": "CPU", "GPU 메모리(GB)": None, "생성 시간(초)": round(int8_dynamic_time, 2), "출력": text_int8[:40]},
    {"방법": "optimum-quanto int8", "실행 위치": "GPU", "GPU 메모리(GB)": round(int8_quanto_mem, 3), "생성 시간(초)": round(quanto8_time, 2), "출력": text_quanto8[:40]},
    {"방법": "optimum-quanto int4", "실행 위치": "-", "GPU 메모리(GB)": None, "생성 시간(초)": None, "출력": "미실행 (컴파일이 기기를 멈추는 것 확인, 위 경고 참고)"},
]

df = pd.DataFrame(rows)
df

## 7. 정리 — Jetson Orin Nano에서 양자화, 무엇을 기억해야 하는가

1. **bitsandbytes 기반 `load_in_8bit`/`load_in_4bit`는 이 하드웨어에서 실패한다** — 작은 토이 텐서 테스트는 통과하지만, 실제 모델 `generate()`에서 `cuBLAS API failed with status 15` 에러가 납니다. sm_87용 사전 컴파일 커널이 없기 때문([issue #1930](https://github.com/bitsandbytes-foundation/bitsandbytes/issues/1930))이며, 소스 빌드나 Jetson AI Lab 커뮤니티 휠 없이는 신뢰할 수 없습니다.
2. **`torch.quantization` dynamic quantization**은 100% 확실히 동작하지만 CPU 전용이라 Jetson GPU 가속의 이점을 못 받는다. 메모리 압박 시 최후 수단.
3. **`optimum-quanto` int8**은 실제로 GPU에서 안정적으로 동작함을 확인했습니다 (GPU 메모리 약 0.72GB, fp16의 ~1GB보다 소폭 감소). 다만 속도는 fp16보다 오히려 느릴 수 있습니다(전용 커널 부재) — 메모리가 우선순위일 때의 선택지입니다.
4. **`optimum-quanto` int4는 이 하드웨어에서 위험합니다** — 최초 실행 시 여러 CUDA 확장을 동시에 JIT 컴파일하면서 8GB 메모리 + 2GB 스왑을 전부 소진해 기기 전체가 멈추는 것을 실제로 겪었습니다. 시도한다면 `MAX_JOBS=1` 같은 안전장치 없이는 절대 시도하지 마세요.
5. **사전 양자화된 GGUF + llama.cpp(CUDA sm_87 빌드)**는 실제 엣지 배포에서 가장 널리 쓰이는 경로이며, 특히 int4가 필요한 7B급 이상 모델에는 사실상 필수적인 선택지다 — 컴파일이 아니라 이미 만들어진 가중치를 다운로드하는 방식이라 위 4번 같은 위험이 없습니다.
6. 8GB급 통합 메모리 엣지 기기에서는 "즉석 컴파일이 필요한 라이브러리"와 "사전 컴파일된 가중치를 그대로 쓰는 방식" 사이의 위험 부담이 크게 다르다는 것 자체가 이 Day의 핵심 교훈입니다.